In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
import tqdm
import numpy as np
from dotenv import load_dotenv

from linalgo.hub import BQClient

from wsd.models import JMDict


In [4]:
load_dotenv()

True

In [ ]:
task_id = os.getenv('LINHUB_TASK')
client = BQClient(task_id)
jmdict = JMDict()

In [7]:
task_id

'823b4545-5c97-4a22-b5f9-1bf75e620e4e'

In [6]:
annotations = client.get_annotations()
docs = client.get_documents()
docs = [doc for doc in docs if len(doc.annotations) > 0]

Forbidden: 403 Access Denied: Table linalgo-infra:linhub_prod.public_linhub_annotation: User does not have permission to query table linalgo-infra:linhub_prod.public_linhub_annotation, or perhaps it does not exist.; reason: accessDenied, message: Access Denied: Table linalgo-infra:linhub_prod.public_linhub_annotation: User does not have permission to query table linalgo-infra:linhub_prod.public_linhub_annotation, or perhaps it does not exist.

Location: US
Job ID: b7f7c53c-7199-45a0-b804-fbe5069299e9


In [ ]:
def get_offset(token):
    start = token.target.selector[0].start_offset
    end = token.target.selector[0].end_offset
    return start, end

y_pred = []
y_true = []
for doc in tqdm.tqdm(docs):
    yp = jmdict.predict(doc.content)
    yp = [y for y in yp if y is not None]
    yt = [a.body for a in sorted(doc.annotations, key=get_offset)]
    # TODO: Will break when more than one annotator
    if len(yt) == len(yp):  # Some documents are partially annotated.
        y_pred.extend(yp)
        y_true.extend(yt)
y_true = np.array(y_true)
y_pred = np.array(y_pred)

In [ ]:
acc = np.sum(y_true == y_pred) / len(y_true)
print(f"accuracy = {acc:%}")